# Visualización de datos de alta dimensionalidad con t-SNE

## Caso de estudio: ventas semanales de Corporación Favorita

**Herramientas:** Polars + pandas + scikit-learn + Plotly  
**Archivo:** `favorita_weekly.parquet`

### Objetivo

Transformar los datos de ventas semanales para que:

- cada fila represente una combinación **tienda + semana**;
- cada columna represente un producto;
- cada punto del mapa t-SNE represente el comportamiento semanal de una tienda;
- las observaciones con patrones de venta localmente similares aparezcan próximas en el mapa.

### Pregunta central

> ¿Cómo podemos visualizar una matriz con miles de variables en un espacio de solo dos dimensiones?

Este laboratorio sigue la misma lógica utilizada en el cuaderno de PCA: partimos del archivo original, revisamos su calidad, construimos la matriz de características, escalamos los datos y aplicamos la técnica de reducción dimensional.

## 1. Instalación de las librerías

Google Colab incluye la mayoría de las librerías necesarias. Instalamos o actualizamos Polars y Plotly para trabajar con el archivo y crear visualizaciones interactivas.

In [1]:
!pip -q install -U polars plotly scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.5/846.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.8 MB/s eta 0:00:00


## 2. Importación de librerías

In [3]:
import os
import time
import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

## 3. Subir el archivo a Google Colab

Sube el archivo:

`favorita_weekly.parquet`

El cuaderno espera encontrarlo en la carpeta `/content/`.

In [4]:
# Descomenta estas líneas si deseas subir el archivo manualmente.

# from google.colab import files
# archivos = files.upload()

## 4. Leer el archivo con Polars

Polars permite trabajar eficientemente con archivos de gran tamaño.

In [6]:
RUTA_ARCHIVO = "/content/favorita_weekly.parquet"

if not os.path.exists(RUTA_ARCHIVO):
    raise FileNotFoundError(
        f"No se encontró el archivo {RUTA_ARCHIVO}. "
        "Súbelo a Google Colab antes de continuar."
    )

df = pl.read_parquet(RUTA_ARCHIVO)
df

fecha_semana,anio,semana,store_nbr,item_nbr,unit_sales,dias_promocion,dias_con_venta
datetime[ns],i16,i8,i16,i32,f32,i8,i8
2012-12-31 00:00:00,2013,1,1,103520,5.0,0,2
2012-12-31 00:00:00,2013,1,1,103665,13.0,0,5
2012-12-31 00:00:00,2013,1,1,105574,26.0,0,5
2012-12-31 00:00:00,2013,1,1,105575,38.0,0,5
2012-12-31 00:00:00,2013,1,1,105577,11.0,0,5
…,…,…,…,…,…,…,…
2017-08-14 00:00:00,2017,33,54,2109909,5.0,0,1
2017-08-14 00:00:00,2017,33,54,2110456,312.0,0,2
2017-08-14 00:00:00,2017,33,54,2113343,1.0,0,1


## 5. Revisar las primeras filas

In [7]:
df.head()

fecha_semana,anio,semana,store_nbr,item_nbr,unit_sales,dias_promocion,dias_con_venta
datetime[ns],i16,i8,i16,i32,f32,i8,i8
2012-12-31 00:00:00,2013,1,1,103520,5.0,0,2
2012-12-31 00:00:00,2013,1,1,103665,13.0,0,5
2012-12-31 00:00:00,2013,1,1,105574,26.0,0,5
2012-12-31 00:00:00,2013,1,1,105575,38.0,0,5
2012-12-31 00:00:00,2013,1,1,105577,11.0,0,5


## 6. Revisar la estructura del dataset

Observamos:

- cantidad de filas y columnas;
- nombres de las variables;
- tipos de datos.

In [8]:
print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns)
print("\nTipos de datos:")
print(df.schema)

Dimensiones: (23582329, 8)

Columnas:
['fecha_semana', 'anio', 'semana', 'store_nbr', 'item_nbr', 'unit_sales', 'dias_promocion', 'dias_con_venta']

Tipos de datos:
Schema({'fecha_semana': Datetime(time_unit='ns', time_zone=None), 'anio': Int16, 'semana': Int8, 'store_nbr': Int16, 'item_nbr': Int32, 'unit_sales': Float32, 'dias_promocion': Int8, 'dias_con_venta': Int8})


## 7. Definir las columnas del archivo

El cuaderno utilizará los siguientes nombres:

- `fecha_semana`: fecha correspondiente al inicio de cada semana;
- `store_nbr`: tienda;
- `item_nbr`: producto;
- `unit_sales`: ventas semanales.

Si el archivo utiliza nombres diferentes, modifica solamente esta celda.

In [9]:
FECHA_SEMANA = "fecha_semana"
TIENDA = "store_nbr"
PRODUCTO = "item_nbr"
VENTAS = "unit_sales"

## 8. Seleccionar las variables necesarias

Conservamos únicamente las cuatro columnas requeridas para construir la matriz de características.

In [10]:
df = df.select([
    pl.col(FECHA_SEMANA).alias("fecha_semana"),
    pl.col(TIENDA).alias("tienda"),
    pl.col(PRODUCTO).alias("producto"),
    pl.col(VENTAS).alias("ventas")
])

df.head()

fecha_semana,tienda,producto,ventas
datetime[ns],i16,i32,f32
2012-12-31 00:00:00,1,103520,5.0
2012-12-31 00:00:00,1,103665,13.0
2012-12-31 00:00:00,1,105574,26.0
2012-12-31 00:00:00,1,105575,38.0
2012-12-31 00:00:00,1,105577,11.0


## 9. Resumen general del dataset

In [11]:
df.select([
    pl.len().alias("registros"),
    pl.col("fecha_semana").n_unique().alias("semanas"),
    pl.col("tienda").n_unique().alias("tiendas"),
    pl.col("producto").n_unique().alias("productos")
])

registros,semanas,tiendas,productos
u32,u32,u32,u32
23582329,242,54,4036


## 10. Revisar valores nulos

In [12]:
df.null_count()

fecha_semana,tienda,producto,ventas
u32,u32,u32,u32
0,0,0,0


## 11. Revisar duplicados

La llave lógica del archivo semanal es:

**fecha_semana + tienda + producto**

Como el archivo ya está consolidado semanalmente, esta combinación debería ser única.

In [13]:
duplicados = (
    df.group_by(["fecha_semana", "tienda", "producto"])
      .len()
      .filter(pl.col("len") > 1)
)

print("Combinaciones duplicadas:", duplicados.height)

Combinaciones duplicadas: 0


## 12. Limpiar y consolidar los datos

- eliminamos filas sin identificadores;
- reemplazamos ventas nulas por cero;
- agrupamos por semana, tienda y producto;
- sumamos posibles registros repetidos.

In [14]:
df_limpio = (
    df.drop_nulls(["fecha_semana", "tienda", "producto"])
      .with_columns(
          pl.col("ventas").fill_null(0).cast(pl.Float32)
      )
      .group_by(["fecha_semana", "tienda", "producto"])
      .agg(
          pl.col("ventas").sum().alias("ventas")
      )
)

df_limpio.head()

fecha_semana,tienda,producto,ventas
datetime[ns],i16,i32,f32
2013-05-27 00:00:00,4,470624,81.0
2016-07-18 00:00:00,35,1083152,13.0
2016-12-26 00:00:00,2,1342009,90.0
2015-08-10 00:00:00,21,1372566,7.0
2013-12-23 00:00:00,2,1045449,9.0


## 13. Definir la unidad de análisis

Cada fila del archivo representa las ventas semanales de un producto en una tienda.

Para describir el comportamiento completo de una tienda necesitamos una nueva unidad de análisis:

> **Tienda + Semana**

In [15]:
observaciones = (
    df_limpio.select(["fecha_semana", "tienda"])
             .unique()
             .height
)

print("Observaciones tienda-semana:", observaciones)

Observaciones tienda-semana: 12060


## 14. Seleccionar los productos más frecuentes

Para controlar el tamaño de la matriz, seleccionaremos hasta 5.000 productos.

Los productos se ordenan según la cantidad de registros en los que aparecen.

In [16]:
MAX_PRODUCTOS = 5000

productos_seleccionados = (
    df_limpio.group_by("producto")
             .len()
             .sort("len", descending=True)
             .head(MAX_PRODUCTOS)
             .get_column("producto")
             .to_list()
)

print("Productos seleccionados:", len(productos_seleccionados))

Productos seleccionados: 4036


## 15. Filtrar los productos seleccionados

In [17]:
df_filtrado = df_limpio.filter(
    pl.col("producto").is_in(productos_seleccionados)
)

df_filtrado.shape

(23582329, 4)

## 16. Construir la matriz de características

Aplicamos un **pivot**:

- filas: `fecha_semana + tienda`;
- columnas: productos;
- valores: ventas.

Los productos dejan de ser registros y pasan a ser características.

In [18]:
matriz = (
    df_filtrado.pivot(
        values="ventas",
        index=["fecha_semana", "tienda"],
        on="producto",
        aggregate_function="sum"
    )
    .fill_null(0)
    .sort(["fecha_semana", "tienda"])
)

matriz.head()

fecha_semana,tienda,470624,1083152,1342009,1372566,1045449,956014,211206,802833,1463867,210798,1464210,732006,514172,1939144,2011218,1239785,1230417,1463569,409738,913964,1423681,115611,457425,1963684,819194,802832,1104599,1253765,1964738,564288,1466827,872317,1248391,818588,268664,…,2010082,2121610,2054300,2049026,2123727,2122868,2122676,2011457,2011468,2116139,2123410,2123791,2035576,2008567,2123839,2120723,2118662,2123747,2123863,2011448,2011459,2126944,2123209,2011470,2116238,2123036,2121690,2123790,2123711,2116132,2123859,2126842,2122818,2122947,2015898,2011451,2123463
datetime[ns],i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
2012-12-31 00:00:00,1,45.0,101.0,0.0,0.0,6.0,36.0,6.0,11.0,0.0,12.0,0.0,13.0,40.0,0.0,0.0,0.0,0.0,0.0,31.0,17.0,0.0,146.0,17.0,0.0,6.0,5.0,12.0,0.0,0.0,33.0,0.0,3.0,0.0,1.0,20.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-12-31 00:00:00,2,134.0,160.0,0.0,0.0,12.0,97.0,8.0,45.0,0.0,11.0,0.0,12.0,68.0,0.0,0.0,0.0,0.0,0.0,29.0,8.0,0.0,131.0,21.0,0.0,12.0,28.0,19.0,0.0,0.0,36.0,0.0,13.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-12-31 00:00:00,3,32.0,335.0,0.0,0.0,6.0,170.0,64.0,213.0,0.0,19.0,0.0,38.0,187.0,0.0,0.0,0.0,0.0,0.0,171.0,36.0,0.0,302.0,66.0,0.0,17.0,127.0,46.0,0.0,0.0,115.0,0.0,30.0,0.0,3.0,124.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-12-31 00:00:00,4,50.0,144.0,0.0,0.0,4.0,139.0,17.0,29.0,0.0,32.0,0.0,16.0,90.0,0.0,0.0,0.0,0.0,0.0,20.0,16.0,0.0,60.0,16.0,0.0,5.0,60.0,29.0,0.0,0.0,26.0,0.0,16.0,0.0,0.0,59.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-12-31 00:00:00,5,66.0,35.0,0.0,0.0,42.0,51.0,3.0,93.0,0.0,18.0,0.0,16.0,59.0,0.0,0.0,0.0,0.0,0.0,30.0,20.0,0.0,115.0,12.0,0.0,23.0,49.0,15.0,0.0,0.0,29.0,0.0,11.0,0.0,1.0,95.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 17. Dimensiones de la matriz

In [19]:
print("Observaciones:", matriz.height)
print("Variables totales:", matriz.width)
print("Productos como características:", matriz.width - 2)

Observaciones: 12060
Variables totales: 4038
Productos como características: 4036


## 18. Separar identificadores y características

Las columnas `fecha_semana` y `tienda` identifican cada observación.

Las columnas de productos forman la matriz de características. Hasta aquí trabajamos con Polars. A partir de este punto convertimos solo la matriz numérica a pandas, porque `scikit-learn` se integra naturalmente con esta estructura.

In [20]:
identificadores = matriz.select(["fecha_semana", "tienda"])

df_features = (
    matriz
    .drop(["fecha_semana", "tienda"])
    .to_pandas()
)

print("Tipo de objeto:", type(df_features))
print("Forma de la matriz de características:", df_features.shape)

df_features.head()

Tipo de objeto: <class 'pandas.core.frame.DataFrame'>
Forma de la matriz de características: (12060, 4036)


,470624,1083152,1342009,1372566,1045449,956014,211206,802833,1463867,210798,...,2123790,2123711,2116132,2123859,2126842,2122818,2122947,2015898,2011451,2123463
0,45.0,101.0,0.0,0.0,6.0,36.0,6.0,11.0,0.0,12.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,134.0,160.0,0.0,0.0,12.0,97.0,8.0,45.0,0.0,11.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,32.0,335.0,0.0,0.0,6.0,170.0,64.0,213.0,0.0,19.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,50.0,144.0,0.0,0.0,4.0,139.0,17.0,29.0,0.0,32.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,66.0,35.0,0.0,0.0,42.0,51.0,3.0,93.0,0.0,18.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 19. Analizar la dispersión de la matriz

En una semana determinada no todos los productos se venden en todas las tiendas. Por eso muchas celdas contienen cero.

In [21]:
porcentaje_ceros = df_features.eq(0).to_numpy().mean() * 100
filas, columnas = df_features.shape

print(f"Observaciones              : {filas:,}")
print(f"Productos                  : {columnas:,}")
print(f"Porcentaje de valores cero : {porcentaje_ceros:.2f}%")
print(f"Densidad de la matriz      : {100 - porcentaje_ceros:.2f}%")

Observaciones              : 12,060
Productos                  : 4,036
Porcentaje de valores cero : 51.55%
Densidad de la matriz      : 48.45%


## 20. Escalar los datos

t-SNE se basa en relaciones de similitud entre observaciones. Si los productos tienen escalas muy diferentes, aquellos con mayores magnitudes pueden dominar el cálculo.

`StandardScaler` transforma cada producto para que tenga media 0 y desviación estándar 1.

In [22]:
scaler = StandardScaler()

df_escalado = pd.DataFrame(
    scaler.fit_transform(df_features),
    columns=df_features.columns,
    index=df_features.index
)

print("Forma de la matriz escalada:", df_escalado.shape)
df_escalado.head()

Forma de la matriz escalada: (12060, 4036)


,470624,1083152,1342009,1372566,1045449,956014,211206,802833,1463867,210798,...,2123790,2123711,2116132,2123859,2126842,2122818,2122947,2015898,2011451,2123463
0,0.181210,0.104735,-0.843072,-0.335688,-0.353352,-0.398021,-0.351458,-0.483424,-0.780878,-0.497564,...,-0.022736,-0.017046,-0.022311,-0.016814,-0.020537,-0.011519,-0.020366,-0.009106,-0.009106,-0.013301
1,2.190633,0.675079,-0.843072,-0.335688,-0.002997,0.350087,-0.288081,-0.021103,-0.780878,-0.529492,...,-0.022736,-0.017046,-0.022311,-0.016814,-0.020537,-0.011519,-0.020366,-0.009106,-0.009106,-0.013301
2,-0.112301,2.366779,-0.843072,-0.335688,-0.353352,1.245364,1.486457,2.263305,-0.780878,-0.274065,...,-0.022736,-0.017046,-0.022311,-0.016814,-0.020537,-0.011519,-0.020366,-0.009106,-0.009106,-0.013301
3,0.294099,0.520410,-0.843072,-0.335688,-0.470137,0.865178,-0.002888,-0.238666,-0.780878,0.141004,...,-0.022736,-0.017046,-0.022311,-0.016814,-0.020537,-0.011519,-0.020366,-0.009106,-0.009106,-0.013301
4,0.655344,-0.533278,-0.843072,-0.335688,1.748778,-0.214060,-0.446522,0.631585,-0.780878,-0.305994,...,-0.022736,-0.017046,-0.022311,-0.016814,-0.020537,-0.011519,-0.020366,-0.009106,-0.009106,-0.013301


## 21. Definir los parámetros de t-SNE

Los principales parámetros que utilizaremos son:

- `n_components=2`: proyectar los datos en dos dimensiones;
- `perplexity=30`: tamaño aproximado de la vecindad considerada por el algoritmo;
- `init="pca"`: inicialización estable;
- `learning_rate="auto"`: tasa de aprendizaje ajustada automáticamente;
- `random_state=42`: resultado reproducible;
- `max_iter=1000`: número máximo de iteraciones.

La perplejidad debe ser menor que el número de observaciones.

In [23]:
PERPLEXITY = 30
SEMILLA = 42

if PERPLEXITY >= len(df_escalado):
    raise ValueError(
        "La perplejidad debe ser menor que el número de observaciones."
    )

tsne = TSNE(
    n_components=2,
    perplexity=PERPLEXITY,
    init="pca",
    learning_rate="auto",
    max_iter=1000,
    random_state=SEMILLA,
    verbose=1
)

## 22. Aplicar t-SNE

t-SNE transforma la matriz de miles de productos en dos coordenadas visuales.

El tiempo de ejecución depende del número de observaciones, variables y recursos disponibles en Google Colab.

In [24]:
inicio = time.time()

coordenadas_tsne = tsne.fit_transform(df_escalado)

duracion = time.time() - inicio

print(f"Tiempo de ejecución: {duracion:.2f} segundos")
print("Forma del resultado:", coordenadas_tsne.shape)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 12060 samples in 0.325s...
[t-SNE] Computed neighbors for 12060 samples in 51.340s...
[t-SNE] Computed conditional probabilities for sample 1000 / 12060
[t-SNE] Computed conditional probabilities for sample 2000 / 12060
[t-SNE] Computed conditional probabilities for sample 3000 / 12060
[t-SNE] Computed conditional probabilities for sample 4000 / 12060
[t-SNE] Computed conditional probabilities for sample 5000 / 12060
[t-SNE] Computed conditional probabilities for sample 6000 / 12060
[t-SNE] Computed conditional probabilities for sample 7000 / 12060
[t-SNE] Computed conditional probabilities for sample 8000 / 12060
[t-SNE] Computed conditional probabilities for sample 9000 / 12060
[t-SNE] Computed conditional probabilities for sample 10000 / 12060
[t-SNE] Computed conditional probabilities for sample 11000 / 12060
[t-SNE] Computed conditional probabilities for sample 12000 / 12060
[t-SNE] Computed conditional probabilities for sa

## 23. Construir el DataFrame de resultados

Las columnas `tSNE1` y `tSNE2` son coordenadas visuales. No representan variables del negocio ni tienen unidades físicas.

In [25]:
df_tsne = pd.DataFrame(
    coordenadas_tsne,
    columns=["tSNE1", "tSNE2"],
    index=df_escalado.index
)

df_tsne["tienda"] = identificadores.get_column("tienda").to_list()
df_tsne["fecha_semana"] = pd.to_datetime(
    identificadores.get_column("fecha_semana").to_list()
)

df_tsne.head()

,tSNE1,tSNE2,tienda,fecha_semana
0,28.022434,-48.009113,1,2012-12-31
1,42.459003,-46.432884,2,2012-12-31
2,55.845531,-80.871330,3,2012-12-31
3,37.905678,-46.650486,4,2012-12-31
4,9.482080,-55.505089,5,2012-12-31


## 24. Visualizar el mapa t-SNE

Cada punto representa una tienda durante una semana.

La interpretación principal es **local**: puntos próximos sugieren patrones de venta similares. Las distancias globales entre grupos deben interpretarse con cautela.

In [43]:
fig = px.scatter(
    df_tsne,
    x="tSNE1",
    y="tSNE2",
    hover_data=["tienda", "fecha_semana"],
    title="Mapa t-SNE de las observaciones tienda-semana",
    labels={
        "tSNE1": "Dimensión t-SNE 1",
        "tSNE2": "Dimensión t-SNE 2"
    },
)

fig.update_traces(
    marker=dict(size=5)
)
fig.update_layout(
    width=1000,
    height=700
)

fig.show()

### Interpretación

- Cada punto representa el comportamiento semanal de una tienda.
- Puntos próximos sugieren vecindades de comportamiento similares.
- Zonas densas pueden indicar patrones frecuentes.
- Puntos aislados pueden corresponder a observaciones atípicas.
- Los ejes no tienen significado físico.
- La distancia entre grupos lejanos no debe utilizarse como una medida exacta.

## 25. Visualizar las observaciones según la tienda

In [29]:
df_tsne["tienda_cat"] = df_tsne["tienda"].astype(str)

fig = px.scatter(
    df_tsne,
    x="tSNE1",
    y="tSNE2",
    color="tienda_cat",
    hover_data=["fecha_semana"],
    title="Mapa t-SNE coloreado por tienda",
    labels={
        "tSNE1": "Dimensión t-SNE 1",
        "tSNE2": "Dimensión t-SNE 2",
        "tienda_cat": "Tienda"
    },
    opacity=0.7
)

fig.update_layout(
    width=1100,
    height=750
)

fig.show()

## 26. Visualizar las observaciones según el año

El color permite evaluar si existe una organización temporal visible en el mapa.

In [30]:
df_tsne["Año"] = df_tsne["fecha_semana"].dt.year.astype(str)

fig = px.scatter(
    df_tsne,
    x="tSNE1",
    y="tSNE2",
    color="Año",
    hover_data=["tienda", "fecha_semana"],
    title="Mapa t-SNE coloreado por año",
    labels={
        "tSNE1": "Dimensión t-SNE 1",
        "tSNE2": "Dimensión t-SNE 2"
    },
    opacity=0.7
)

fig.update_layout(
    width=1000,
    height=700
)

fig.show()

### Interpretación por año

La existencia de zonas dominadas por determinados años sugeriría cambios temporales en los patrones de venta. Una amplia superposición indicaría que el año, por sí solo, no explica completamente la estructura local observada.

La interpretación final debe realizarse después de ejecutar el cuaderno y observar el gráfico real.

## 27. Visualizar las observaciones según el mes

Esta visualización permite explorar posibles patrones estacionales.

In [32]:
meses = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}

df_tsne["Mes"] = df_tsne["fecha_semana"].dt.month.map(meses)
orden_meses = list(meses.values())

fig = px.scatter(
    df_tsne,
    x="tSNE1",
    y="tSNE2",
    color="Mes",
    category_orders={"Mes": orden_meses},
    hover_data=["tienda", "fecha_semana"],
    title="Mapa t-SNE coloreado por mes",
    labels={
        "tSNE1": "Dimensión t-SNE 1",
        "tSNE2": "Dimensión t-SNE 2"
    },
    opacity=0.7
)

fig.update_layout(
    width=1100,
    height=750,
    legend_title="Mes"
)

fig.show()

### Interpretación por mes

- Los meses aparecen ampliamente mezclados en las diferentes regiones del mapa t-SNE.
- No se observa una separación clara por mes, por lo que la estacionalidad mensual no parece ser el principal factor que organiza las observaciones.
- La estructura encontrada podría estar asociada principalmente a características propias de las tiendas y a sus patrones de venta.

## 28. Analizar observaciones potencialmente aisladas

Como aproximación exploratoria, calculamos la distancia de cada punto a su vecino más cercano en el mapa 2D.

Una distancia alta puede señalar una observación visualmente aislada. Esto no reemplaza un método formal de detección de anomalías.

In [33]:
distancias_2d = pairwise_distances(
    df_tsne[["tSNE1", "tSNE2"]]
)

np.fill_diagonal(distancias_2d, np.inf)

df_tsne["distancia_vecino_mas_cercano"] = (
    distancias_2d.min(axis=1)
)

posibles_aislados = (
    df_tsne.sort_values(
        "distancia_vecino_mas_cercano",
        ascending=False
    )
    .head(10)
)

posibles_aislados[
    [
        "tienda",
        "fecha_semana",
        "tSNE1",
        "tSNE2",
        "distancia_vecino_mas_cercano"
    ]
]

,tienda,fecha_semana,tSNE1,tSNE2,distancia_vecino_mas_cercano
10168,35,2016-12-12,-17.164265,-6.459835,5.475537
4765,35,2014-12-08,-22.161720,-8.697455,5.441743
11988,37,2017-08-07,38.932407,28.023304,4.168474
7436,35,2015-12-14,-2.252884,28.111343,3.264296
9204,1,2016-08-08,26.692142,7.760958,2.528891
4805,26,2014-12-15,0.878947,3.567736,2.512599
4800,18,2014-12-15,23.220547,46.043976,2.282581
6418,13,2015-08-03,-41.790199,-36.983490,2.216255
12027,22,2017-08-14,-12.448092,10.707406,2.171326
10907,32,2017-03-20,-18.842009,20.290585,1.837109


## 29. Sensibilidad a la perplejidad

t-SNE puede producir mapas diferentes cuando cambia la perplejidad.

La siguiente celda es **opcional**, porque ejecuta t-SNE varias veces y puede tardar algunos minutos. Su propósito es comprobar que las conclusiones no dependan de una única configuración.

In [34]:
# EJECUCIÓN OPCIONAL

PERPLEJIDADES = [10, 30, 50]
resultados_perplejidad = {}

for p in PERPLEJIDADES:
    if p >= len(df_escalado):
        continue

    modelo = TSNE(
        n_components=2,
        perplexity=p,
        init="pca",
        learning_rate="auto",
        max_iter=1000,
        random_state=SEMILLA,
        verbose=0
    )

    resultados_perplejidad[p] = modelo.fit_transform(df_escalado)

print("Perplejidades ejecutadas:", list(resultados_perplejidad))

Perplejidades ejecutadas: [10, 30, 50]


## 30. Visualizar los resultados de distintas perplejidades

Cada gráfico debe analizarse por separado. No debemos esperar que los ejes o la orientación coincidan entre ejecuciones.

In [35]:
# EJECUTAR DESPUÉS DE LA CELDA ANTERIOR

for p, coordenadas in resultados_perplejidad.items():
    df_temp = pd.DataFrame(
        coordenadas,
        columns=["tSNE1", "tSNE2"]
    )

    fig = px.scatter(
        df_temp,
        x="tSNE1",
        y="tSNE2",
        title=f"Mapa t-SNE con perplejidad = {p}",
        opacity=0.65
    )

    fig.update_layout(
        width=900,
        height=650
    )

    fig.show()

## 31. Comparar PCA y t-SNE

PCA y t-SNE no son algoritmos rivales:

- **PCA** busca una representación lineal que conserve la mayor varianza posible.
- **t-SNE** busca una representación no lineal orientada a conservar vecindades locales.

La comparación visual permite observar cómo cada técnica representa la misma matriz.

In [36]:
pca_2d = PCA(n_components=2)

coordenadas_pca = pca_2d.fit_transform(df_escalado)

df_pca_2d = pd.DataFrame(
    coordenadas_pca,
    columns=["PC1", "PC2"]
)

df_pca_2d["tienda"] = identificadores.get_column("tienda").to_list()
df_pca_2d["fecha_semana"] = pd.to_datetime(
    identificadores.get_column("fecha_semana").to_list()
)

print(
    "Varianza conservada por PC1 y PC2:",
    f"{pca_2d.explained_variance_ratio_.sum():.2%}"
)

Varianza conservada por PC1 y PC2: 30.53%


In [37]:
fig = px.scatter(
    df_pca_2d,
    x="PC1",
    y="PC2",
    hover_data=["tienda", "fecha_semana"],
    title="Las mismas observaciones proyectadas mediante PCA",
    labels={
        "PC1": "Componente principal 1",
        "PC2": "Componente principal 2"
    },
    opacity=0.65
)

fig.update_layout(
    width=1000,
    height=700
)

fig.show()

### Comparación conceptual

| Aspecto | PCA | t-SNE |
|---|---|---|
| Naturaleza | Lineal | No lineal |
| Objetivo | Conservar varianza | Conservar vecindades locales |
| Salida habitual | Varios componentes | 2 o 3 dimensiones |
| Uso principal | Preprocesamiento y modelado | Exploración visual |
| Significado de los ejes | Componentes con varianza explicada | Sin significado físico |
| Entrada para modelos | Sí | Generalmente no |

## 32. PCA y t-SNE dentro del mismo pipeline

En datasets grandes también puede utilizarse el flujo:

**Matriz escalada → PCA → t-SNE**

PCA reduce primero el número de variables y t-SNE construye después el mapa bidimensional. Esta etapa es opcional y se utiliza principalmente para acelerar el procesamiento.

In [38]:
# EJECUCIÓN OPCIONAL: PCA COMO ACELERACIÓN PREVIA

N_COMPONENTES_PREVIOS = min(
    50,
    df_escalado.shape[0] - 1,
    df_escalado.shape[1]
)

pca_previo = PCA(
    n_components=N_COMPONENTES_PREVIOS,
    random_state=SEMILLA
)

datos_pca_previos = pca_previo.fit_transform(df_escalado)

tsne_sobre_pca = TSNE(
    n_components=2,
    perplexity=PERPLEXITY,
    init="pca",
    learning_rate="auto",
    max_iter=1000,
    random_state=SEMILLA,
    verbose=1
)

coordenadas_pca_tsne = tsne_sobre_pca.fit_transform(
    datos_pca_previos
)

print("Matriz original escalada:", df_escalado.shape)
print("Entrada de t-SNE después de PCA:", datos_pca_previos.shape)
print("Mapa final:", coordenadas_pca_tsne.shape)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 12060 samples in 0.002s...
[t-SNE] Computed neighbors for 12060 samples in 2.102s...
[t-SNE] Computed conditional probabilities for sample 1000 / 12060
[t-SNE] Computed conditional probabilities for sample 2000 / 12060
[t-SNE] Computed conditional probabilities for sample 3000 / 12060
[t-SNE] Computed conditional probabilities for sample 4000 / 12060
[t-SNE] Computed conditional probabilities for sample 5000 / 12060
[t-SNE] Computed conditional probabilities for sample 6000 / 12060
[t-SNE] Computed conditional probabilities for sample 7000 / 12060
[t-SNE] Computed conditional probabilities for sample 8000 / 12060
[t-SNE] Computed conditional probabilities for sample 9000 / 12060
[t-SNE] Computed conditional probabilities for sample 10000 / 12060
[t-SNE] Computed conditional probabilities for sample 11000 / 12060
[t-SNE] Computed conditional probabilities for sample 12000 / 12060
[t-SNE] Computed conditional probabilities for sam

## 33. Visualizar el pipeline PCA + t-SNE

In [39]:
df_pca_tsne = pd.DataFrame(
    coordenadas_pca_tsne,
    columns=["tSNE1", "tSNE2"]
)

df_pca_tsne["tienda"] = identificadores.get_column("tienda").to_list()
df_pca_tsne["fecha_semana"] = pd.to_datetime(
    identificadores.get_column("fecha_semana").to_list()
)

fig = px.scatter(
    df_pca_tsne,
    x="tSNE1",
    y="tSNE2",
    hover_data=["tienda", "fecha_semana"],
    title="Mapa obtenido con el pipeline PCA + t-SNE",
    opacity=0.65
)

fig.update_layout(
    width=1000,
    height=700
)

fig.show()

## 34. Exportar el resultado t-SNE

Guardamos las coordenadas junto con los identificadores.

Este archivo sirve para visualización y análisis exploratorio. No debe interpretarse como una nueva matriz de variables para entrenar modelos predictivos.

In [40]:
resultado_tsne = pl.from_pandas(
    df_tsne[
        [
            "fecha_semana",
            "tienda",
            "tSNE1",
            "tSNE2",
            "distancia_vecino_mas_cercano"
        ]
    ]
)

resultado_tsne.write_csv("favorita_tsne.csv")
resultado_tsne.write_parquet("favorita_tsne.parquet")

print("Archivos exportados correctamente.")

Archivos exportados correctamente.


## 35. Comparar el tamaño de los archivos exportados

In [41]:
csv_size = os.path.getsize("favorita_tsne.csv") / (1024**2)
parquet_size = os.path.getsize("favorita_tsne.parquet") / (1024**2)

reduccion = 100 * (1 - parquet_size / csv_size)

print(f"CSV     : {csv_size:.2f} MB")
print(f"Parquet : {parquet_size:.2f} MB")
print(f"Reducción de tamaño: {reduccion:.1f}%")

CSV     : 0.74 MB
Parquet : 0.12 MB
Reducción de tamaño: 83.6%


## 36. Descargar los resultados

In [42]:
from google.colab import files

files.download("favorita_tsne.csv")
files.download("favorita_tsne.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Conclusiones

En este laboratorio construimos un mapa bidimensional a partir de un conjunto de datos retail de alta dimensionalidad.

El proceso desarrollado fue:

- cargar y revisar el archivo semanal de Favorita;
- comprobar valores nulos y combinaciones duplicadas;
- definir la unidad de análisis como **tienda + semana**;
- construir mediante Pivot una matriz donde cada producto es una característica;
- analizar la dispersión de la matriz;
- estandarizar las variables con `StandardScaler`;
- aplicar t-SNE directamente sobre la matriz escalada;
- visualizar las observaciones por tienda, año y mes;
- explorar observaciones potencialmente aisladas;
- analizar opcionalmente la sensibilidad a la perplejidad;
- comparar PCA y t-SNE;
- mostrar que ambas técnicas también pueden utilizarse dentro del mismo pipeline.

## Lección central

> PCA y t-SNE resuelven problemas distintos. PCA produce una representación compacta útil para el modelado; t-SNE produce un mapa orientado a explorar la estructura local de los datos.

t-SNE es una herramienta de **exploración**. Sus mapas permiten generar hipótesis, pero los grupos y anomalías observados deben validarse con análisis adicionales.